<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Entraînez votre détecteur d'objets

Dans ce notebook, vous allez :

1. Installer et configurer [SAM3](https://github.com/facebookresearch/sam3) – un modèle de base qui étiquette automatiquement vos images.
2. Exécuter SAM3 pour générer des étiquettes de boîtes englobantes au format YOLO pour votre jeu de données.
3. Entraîner un modèle de détection d'objets [YOLOv11n](https://docs.ultralytics.com/) sur ces étiquettes.
4. Analyser les résultats de l'entraînement (courbe PR, matrice de confusion).
5. Exporter le modèle entraîné au format ONNX pour le déployer sur votre Duckiebot.

**Avant de commencer :** assurez-vous d'avoir exécuté le [notebook de configuration](../02-Setup-Data-Collection/setup.ipynb) et que le répertoire `assets/data/duckietown_dataset/` contient vos ensembles d'images `train/` et `val/`.

## Authentification Hugging Face

### Créer un compte sur Hugging Face

Si vous n'en avez pas encore, vous devez [créer un compte sur Hugging Face](https://huggingface.co/join).

### Demande d'accès au modèle SAM3

Si vous ne l'avez pas déjà fait, vous devez demander l'accès au modèle SAM3 via Facebook.

Consultez la [page du modèle](https://huggingface.co/facebook/sam3) pour demander l'accès et confirmer son approbation. L'approbation de votre demande d'accès peut prendre quelques minutes.

### Set up your Hugging Face Access Token

Accédez aux [paramètres des jetons Hugging Face](https://huggingface.co/settings/tokens) et cliquez `Create new token`. Dans les paramètres du jeton, vous pouvez choisir `Fine Grained` et assurez-vous que la boîte  `Read access to contents of all public gated repos you can access` est coché. Once you have created your token, click `Copy` button to copy it to the clipboard. 

Vous devez maintenant ouvrir un terminal dans cette fenêtre VSCode (en utilisant le menu de gauche) puis exécuter la commande :

```{bash}
hf auth login
```

Vous pouvez ensuite coller votre jeton et appuyer sur [ENTRÉE]. Vous devriez obtenir un résultat similaire à celui-ci :

    
    Add token as git credential? [y/N]: y
    Token is valid (permission: fineGrained).
    The token `TOKEN_NAME` has been saved to /home/vsuser/.cache/huggingface/stored_tokens
    Cannot authenticate through git-credential as no helper is defined on your machine.
    You might have to re-authenticate when pushing to the Hugging Face Hub.
    Run the following command in your terminal in case you want to set the 'store' credential helper as default.

    git config --global credential.helper store

    Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
    Token has not been saved to git credential helper.
    Your token has been saved to /home/vsuser/.cache/huggingface/token
    Login successful.
    The current active token is: `TOKEN_NAME`
    
Vous pouvez également coller votre jeton (from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)) dans la cellule de code ci-dessous. Vous devriez voir un `token is valid` imprimé.

In [ ]:
from huggingface_hub import HfApi

token = "your_token_here"
api = HfApi()

try:
    user_info = api.whoami(token=token)
    print("Token is valid. Logged in as:", user_info["name"])
except Exception as e:
    print("Token is invalid or expired:", e)

## Dataset

Votre dataset se trouve dans `assets/data/duckietown_dataset/` et a été créé par le [notebook "setup"](../02-Setup-Data-Collection/setup.ipynb). Il devrait déjà contenir deux sous-répertoires :

```
duckietown_dataset/
  train/
    images/   ← your training images
    labels/   ← will be filled in by SAM3 auto-labeling below
  val/
    images/   ← your validation images
    labels/   ← will be filled in by SAM3 auto-labeling below
```

Les fichiers d'étiquettes (`.txt`) n'existent pas encore — SAM3 les créera automatiquement lors de l'étape d'étiquetage automatique.

In [ ]:
import os
import tempfile
import shutil
from typing import List
from datetime import datetime


def zip_sub_dirs(abs_root_dir: str, lst_rel_subdirs: List[str], output_basename: str) -> str:
    """Compressez certains sous-répertoires et renvoyez le chemin du fichier compressé."""
    out_full = f"{output_basename}.zip"
    if os.path.exists(out_full):
        print(f"Le fichier existe déjà à cet emplacement.: {out_full}")
        print("Renommer/Déplacer vers le répertoire d'exécution.\nAucune opération effectuée.")
        return ""
    tmp_dir = tempfile.mkdtemp()
    print(f"[{datetime.now()}] Répertoire temporaire créé à: {tmp_dir}")
    original_paths = [os.path.join(abs_root_dir, _d) for _d in lst_rel_subdirs]
    tmp_paths = [os.path.join(tmp_dir, _d) for _d in lst_rel_subdirs]
    print(f"[{datetime.now()}] Liste des répertoires à inclure dans le fichier zip :")
    for subdir in original_paths:
        assert os.path.exists(subdir), f"Le chemin spécifié n'existe pas : {subdir}\nAbandonner!"
        print(f" - {subdir}")
    print(f"[{datetime.now()}] Déplacer les sous-répertoires vers le répertoire racine temporaire")
    for ori, tmp in zip(original_paths, tmp_paths):
        shutil.move(ori, tmp)
    print(f"[{datetime.now()}] Compression et création de l'archive...")
    ret = shutil.make_archive(output_basename, 'zip', tmp_dir)
    print(f"[{datetime.now()}] Déplacer les sous-répertoires à leur emplacement d'origine")
    for tmp, ori in zip(tmp_paths, original_paths):
        shutil.move(tmp, ori)
    print(f"[{datetime.now()}] Terminé. Archive créée à : {ret}")
    return ret


# REMARQUE : NE PAS modifier 
ZIPPED_DATASET_BASENAME_FILE = "duckietown_dataset"
DATASET_DIR_ZIP = "../../assets/data/duckietown_dataset"
ZIPPED_DATASET_BASENAME_FULL = os.path.join(DATASET_DIR_ZIP, ZIPPED_DATASET_BASENAME_FILE)
TRAIN_DIR_ZIP = "train"
VALIDATION_DIR = "val"

_ = zip_sub_dirs(
    abs_root_dir=DATASET_DIR_ZIP,
    lst_rel_subdirs=[TRAIN_DIR_ZIP, VALIDATION_DIR],
    output_basename=ZIPPED_DATASET_BASENAME_FULL,
)


Si tout s'est bien passé, vous devriez voir le résultat suivant :

```
Terminé. Archive créée à : ../../assets/data/duckietown_dataset/duckietown_dataset.zip
```


## Configuration de l'environnement

Les cellules ci-dessous installent [SAM3](https://github.com/facebookresearch/sam3) (Segment Anything Model 3) directement depuis le clone local situé dans `assets/sam3/`. SAM3 est un modèle de vision et de langage développé par Meta, capable de localiser et de segmenter des objets dans des images à partir d'une instruction textuelle. Nous l'utilisons pour générer automatiquement des annotations de boîtes englobantes pour votre jeu de données, sans aucune annotation manuelle.

La première installation peut prendre quelques minutes. Si SAM3 est déjà installé, l'installation sera ignorée lors des exécutions suivantes.

In [ ]:
import sys, re
from pathlib import Path

# Install dependencies
SAM3_DIR = Path("../../assets/sam3")
if not SAM3_DIR.exists():
    !git clone -q https://github.com/facebookresearch/sam3.git {SAM3_DIR}

pyproject = SAM3_DIR / "pyproject.toml"
pyproject.write_text(re.sub(r'"numpy==[^"]+",?\s*', "", pyproject.read_text(encoding="utf-8")), encoding="utf-8")

for py_file in SAM3_DIR.rglob("*.py"):
    text = py_file.read_text(encoding="utf-8")
    if "from __future__ import annotations" not in text:
        py_file.write_text("from __future__ import annotations\n" + text, encoding="utf-8")

!{sys.executable} -m pip install -q "{SAM3_DIR}"

sys.modules.pop("sam3", None)
if str(SAM3_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(SAM3_DIR.resolve()))


### Vérification du GPU

L'inférence SAM3 est nettement plus rapide sur GPU : comptez **environ 2 à 5 secondes par image sur CPU** contre **environ 0,1 à 0,5 seconde sur GPU**. Pour un jeu de données de 1 000 images, cela représente une différence d'environ 1 heure à environ 5 minutes. L'entraînement est également beaucoup plus rapide avec un GPU.

Si aucun GPU n'est détecté, l'entraînement fonctionnera tout de même, mais sera plus lent.

In [ ]:
# Vérification du GPU
import torch
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("AVERTISSEMENT : Aucun GPU détecté — l’entraînement sera lent sur le CPU.")


## Chemins d'accès et importations de dataset

Ces chemins pointent vers l'ensemble de données créé par le notebook d'installation. Si les assertions ci-dessous échouent, veuillez exécuter d'abord le [notebook "Setup"](../02-Setup-Data-Collection/setup.ipynb).

In [ ]:
# Chemins
import os, yaml
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from ultralytics import YOLO

ASSETS_DIR   = (Path(os.getcwd()) / "../../assets").resolve()
DATASET_DIR  = ASSETS_DIR / "data" / "duckietown_dataset"
TRAIN_DIR    = DATASET_DIR / "train"
VAL_DIR      = DATASET_DIR / "val"
CLASSES_YAML = DATASET_DIR / "classes.yaml"

assert TRAIN_DIR.exists(), f"Répertoire du train introuvable: {TRAIN_DIR}\nRun the Setup notebook first."
assert VAL_DIR.exists(),   f"Répertoire Val introuvable: {VAL_DIR}\nRun the Setup notebook first."

print(f"Train images : {len(list((TRAIN_DIR / 'images').glob('*')))}")
print(f"Val images   : {len(list((VAL_DIR / 'images').glob('*')))}")


## Classes de configuration à détecter

Cela crée un fichier `classes.yaml` qui indique à YOLO l'emplacement de vos données et les classes à détecter. Le dictionnaire `names` associe les identifiants de classe numériques à des libellés explicites.

Ajoutez des entrées supplémentaires si vous souhaitez détecter d'autres objets, par exemple :

```yaml
names:
  0: 'yellow rubber duck'
  1: 'cone'
  2: 'duckiebot'
```


In [ ]:
# Write YOLO config
CLASSES_YAML.write_text(f"""train: {TRAIN_DIR}
val:   {VAL_DIR}

names:
  0: 'yellow rubber duck'  # Ajoutez d'autres classes ici si nécessaire.
""")
print(CLASSES_YAML.read_text())


## Étiquetage automatique avec SAM3

SAM3 parcourra chaque image de vos ensembles `train/` et `val/` et générera un fichier d'étiquettes `.txt` pour chacune d'elles. Chaque ligne d'un fichier d'étiquettes correspond à un objet détecté et suit le format YOLO :

```
class_id  cx  cy  width  height
```

où toutes les valeurs sont normalisées à `[0, 1]` par rapport aux dimensions de l'image.

**Seuil de confiance :** les détections inférieures à `confidence_threshold=0.3` sont ignorées. Augmentez cette valeur si SAM3 génère trop de faux positifs ; diminuez-la s’il manque des objets.

> **Conseil :** Si l’exécution est interrompue, vous pouvez relancer cette cellule sans risque ; elle ignorera les images qui possèdent déjà un fichier d’étiquette.

In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor


def load_classes(classes_yaml):
    with open(classes_yaml) as f:
        cfg = yaml.safe_load(f)
    return [name for _, name in sorted(cfg["names"].items(), key=lambda kv: int(kv[0]))]


def xyxy_to_yolo_line(bbox, w, h, class_id):
    x1, y1, x2, y2 = bbox
    cx, cy = (x1 + x2) / 2 / w, (y1 + y2) / 2 / h
    bw, bh = (x2 - x1) / w,     (y2 - y1) / h
    return f"{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"


class Sam3AutoLabel:
    def __init__(self, train_dir, val_dir, classes_yaml, confidence_threshold=0.3, device=None):
        self.train_dir = train_dir
        self.val_dir   = val_dir
        self.classes   = load_classes(classes_yaml)
        self.device    = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"SAM3 device: {self.device}")

        bpe = SAM3_DIR / "sam3" / "assets" / "bpe_simple_vocab_16e6.txt.gz"
        self.model = build_sam3_image_model(bpe_path=str(bpe), compile=False)
        self.model.to(self.device).eval()
        self.processor = Sam3Processor(self.model,
                                        confidence_threshold=confidence_threshold,
                                        device=self.device)

    def _detect(self, img_path):
        image = Image.open(img_path).convert("RGB")
        w, h  = image.size
        ctx   = (torch.autocast("cuda", dtype=torch.bfloat16)
                 if self.device == "cuda"
                 else __import__("contextlib").nullcontext())
        dets  = []
        with torch.no_grad(), ctx:
            st = self.processor.set_image(image)
            for class_id, prompt in enumerate(self.classes):
                out    = self.processor.set_text_prompt(state=st, prompt=prompt)
                boxes  = out.get("boxes")
                scores = out.get("scores")
                if boxes is None:
                    continue
                for b, s in zip(boxes, scores):
                    s = float(s)
                    if s >= self.processor.confidence_threshold:
                        dets.append({"bbox": b.tolist(), "score": s, "class_id": class_id})
        return dets, (w, h)

    def _label_split(self, split):
        split_dir = self.train_dir if split == "train" else self.val_dir
        img_dir   = split_dir / "images"
        lbl_dir   = split_dir / "labels"
        imgs = [p for ext in ("*.jpg", "*.jpeg", "*.png") for p in img_dir.rglob(ext)]
        for img_path in tqdm(imgs, desc=f"Labeling {split}"):
            dets, (w, h) = self._detect(img_path)
            lines = [xyxy_to_yolo_line(d["bbox"], w, h, d["class_id"]) for d in dets]
            lbl   = lbl_dir / f"{img_path.stem}.txt"
            if lines:
                lbl.write_text("\n".join(lines))
            elif lbl.exists():
                lbl.unlink()

    def run(self):
        self._label_split("train")
        self._label_split("val")


autolabel = Sam3AutoLabel(
    train_dir=TRAIN_DIR,
    val_dir=VAL_DIR,
    classes_yaml=CLASSES_YAML,
    confidence_threshold=0.3,
)
autolabel.run()

## Vérifier les étiquettes

Avant l'entraînement, il est conseillé de vérifier visuellement que SAM3 produit des étiquettes pertinentes. La cellule ci-dessous sélectionne la première image étiquetée de votre ensemble d'entraînement et y trace les cadres de délimitation prédits.

Vérifiez que :

- Les cadres entourent précisément les objets d'intérêt

- Il n'y a pas de faux positifs évidents (cadres sur fond clair)

- Les étiquettes de classe sont correctes

Si les étiquettes semblent de mauvaise qualité, essayez d'ajuster le seuil de confiance (`confidence_threshold`) dans la cellule d'étiquetage automatique ci-dessus et relancez le processus.

In [ ]:
train_labels = TRAIN_DIR / "labels"
train_images = TRAIN_DIR / "images"
label_files  = list(train_labels.glob("*.txt"))
assert label_files, f"No label files in {train_labels}"

label = label_files[0]
img   = next(
    (train_images / f"{label.stem}{ext}" for ext in (".png", ".jpg", ".jpeg")
     if (train_images / f"{label.stem}{ext}").exists()),
    None,
)
assert img, f"Image not found for label {label.name}"

image  = Image.open(img).convert("RGB")
w, h   = image.size
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image)
ax.axis("off")
classes = load_classes(CLASSES_YAML)
for line in label.read_text().splitlines():
    parts = line.split()
    if len(parts) != 5:
        continue
    cid, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
    x1, y1 = (cx - bw / 2) * w, (cy - bh / 2) * h
    ax.add_patch(plt.Rectangle((x1, y1), bw * w, bh * h, fill=False, linewidth=2))
    ax.text(x1, y1 - 2, classes[cid] if cid < len(classes) else str(cid),
            fontsize=10, bbox=dict(facecolor="black", alpha=0.5), color="white")
plt.show()


## Entraînement

Nous entraînons un modèle **YOLOv11 nano** (`yolo11n.yaml`) - la variante la plus petite et la plus rapide, bien adaptée au déploiement sur le matériel d'un Duckiebot.

Key parameters:

| Parameter | Value | Notes |
|-----------|-------|-------|
| `epochs` | 10 | Nombre maximal d'époques d'entraînement - augmenter pour une meilleure précision |
| `imgsz` | (480, 640) | Do correspondre à la résolution de votre camèra |
| `batch` | 16 | Réduisez à 8 si vous manquez de mémoire GPU |
| `workers` | 0 | Définissez cette valeur à 0 pour éviter les erreurs de mémoire partagée dans Docker. |

Les résultats de l'entraînement (poids, graphiques, métriques) sont enregistrés dans `assets/data/duckietown_dataset/runs/latest`.

In [ ]:
# Train - Ajuster les époques et les lots selon les besoins
model = YOLO("yolo11n.yaml")
model.train(
    data=str(CLASSES_YAML),
    epochs=10,
    imgsz=(480, 640),
    batch=16,
    workers=0,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(DATASET_DIR / "runs"),
    name="duckietown_detection",
    rect=True, # pas de gaspillage de rembourrage sur non carrées 480x640
)

## Exporter au format ONNX

ONNX (Open Neural Network Exchange) est un format de modèle portable qui peut s'exécuter sur de nombreux environnements d'exécution sans nécessiter PyTorch. Nous exportons au format ONNX afin que le modèle puisse s'exécuter efficacement sur Duckiebot grâce à `onnxruntime`.

Le modèle exporté sera enregistré dans `assets/best.onnx` ; c’est le fichier que charge le [notebook d’intégration](../04-Integration/integration.ipynb).

Paramètres d'exportation:
- `opset=15` - ONNX opset version
- `simplify=True` - exécute ONNX "simplifier" pour réduire la taille du modèle
- `nms=True` - La suppression des non-maximums est intégrée au graphe, la sortie est donc déjà filtrée ; sinon, vous pouvez implémenter la suppression des non-maximums vous-même et l'utiliser comme étape de post-traitement.
- `half=True` - FP16 poids si GPU disponible (plus petit, plus rapide)
- Output shape: `[1, 300, 6]` → jusqu'à 300 détections, chacune avec `(x1, y1, x2, y2, score, class_id)`

In [ ]:
# Exportation au format ONNX - enregistrée directement dans assets/best.onnx
runs     = sorted((DATASET_DIR / "runs").iterdir(), key=lambda p: p.stat().st_mtime)
last_run = runs[-1]
best_pt  = last_run / "weights" / "best.pt"

if not best_pt.exists():
    raise FileNotFoundError(f"best.pt not found at {best_pt}")

use_gpu = torch.cuda.is_available()
model   = YOLO(str(best_pt))
model.export(
    format="onnx",
    opset=15,
    imgsz=(480, 640),
    simplify=True,
    dynamic=False,
    nms=True,
    half=use_gpu,
    batch=1,
    device=0 if use_gpu else "cpu",
)

onnx_src = last_run / "weights" / "best.onnx"
onnx_dst = ASSETS_DIR / "best.onnx"
onnx_dst.write_bytes(onnx_src.read_bytes())
print(f"Modèle enregistré dans: {onnx_dst}")
print("Terminé ! Passez à l'étape suivante.")

import shutil
shutil.copytree(last_run, f"{DATASET_DIR}/runs/latest")


# Débogage et inspection du modèle

Une fois l'entraînement terminé, de nombreux résultats intéressants seront générés et pourront vous être utiles.

* Après l'entraînement, un répertoire `runs` est créé sous `assets/data/duckietown_dataset/`. La dernière exécution est systématiquement copiée dans le dossier `assets/data/duckietown_dataset/runs/latest` pour plus de commodité.

* Vous pouvez toujours consulter vos anciennes exécutions. Elles se trouvent dans des dossiers nommés `runs/duckietown_detectionX/`, où `X` est incrémenté à chaque entraînement. Vous pouvez ainsi revenir à un modèle antérieur si nécessaire.

Vous y trouverez notamment votre courbe PR.

<img src="../../assets/data/duckietown_dataset/runs/latest/BoxPR_curve.png" alt="PR Curve" width="50%">

Votre matrice de confusion (que vous indique-t-elle ?) :

<img src="../../assets/data/duckietown_dataset/runs/latest/confusion_matrix.png" alt="PR Curve" width="50%">

And sample training outputs:

<img src="../../assets/data/duckietown_dataset/runs/latest/train_batch1.jpg" alt="PR Curve" width="50%">

Vous pouvez maintenant passer au [notebook d'intégration](../04-Integration/integration.ipynb)!